# 🤖 ADMET Prediction with Machine Learning

This notebook trains a simple ML model to predict aqueous solubility (a key ADMET property).

**You will learn:**
1. Build a small ADMET dataset
2. Generate Morgan fingerprints as features
3. Train a Random Forest model
4. Evaluate predictions
5. Inspect important descriptors


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdFingerprintGenerator
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

print("Libraries loaded.")


## 1. Synthetic ESOL-style dataset

ESOL (Delaney 2004) is the classical aqueous solubility dataset (~1100 molecules with measured logS).

For didactic purposes, here we use a small embedded subset. In a real project you would download the full dataset from MoleculeNet.


In [ ]:
# Small ESOL-style subset (SMILES + measured logS in mol/L)
data = [
    ("CCO",                         -0.77),  # Ethanol
    ("CC(=O)O",                     -0.05),  # Acetic acid
    ("c1ccccc1",                    -2.13),  # Benzene
    ("CCCCCCCC",                    -5.24),  # Octane
    ("CC(=O)Oc1ccccc1C(=O)O",       -2.22),  # Aspirin
    ("CN1C=NC2=C1C(=O)N(C(=O)N2C)C",-1.39),  # Caffeine
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O",  -3.97),  # Ibuprofen
    ("CC(=O)Nc1ccc(O)cc1",          -1.03),  # Paracetamol
    ("CCCCCCCCCCCCCCCC",            -8.40),  # Hexadecane
    ("OCC(O)C(O)C(O)C(O)CO",         0.61),  # Sorbitol
    ("c1ccc2ccccc2c1",              -3.60),  # Naphthalene
    ("Clc1ccccc1",                  -2.77),  # Chlorobenzene
    ("CC(C)O",                       0.43),  # Isopropanol
    ("CCCCO",                       -0.40),  # Butanol
    ("c1ccc(cc1)C(=O)O",            -1.86),  # Benzoic acid
    ("Nc1ccccc1",                   -0.96),  # Aniline
    ("Oc1ccccc1",                   -0.18),  # Phenol
    ("CCN(CC)CC",                    0.83),  # Triethylamine
    ("c1ccc(cc1)c1ccccc1",          -3.98),  # Biphenyl
    ("CC(=O)C",                      0.49),  # Acetone
    ("CCCC(=O)O",                   -0.18),  # Butyric acid
    ("Nc1ccc(N)cc1",                -0.40),  # Phenylenediamine
    ("Oc1ccc(cc1)C(=O)O",           -1.90),  # 4-hydroxybenzoic acid
    ("Clc1ccc(Cl)cc1",              -3.27),  # 1,4-dichlorobenzene
    ("c1ccc(cc1)Cc1ccccc1",         -4.62),  # Diphenylmethane
    ("CCCCCCCCO",                   -2.39),  # Octanol
    ("CCCCCCO",                     -1.24),  # Hexanol
    ("CCCCCCCCCCCCO",               -4.40),  # Dodecanol
    ("CCCCCC",                      -3.66),  # Hexane
    ("c1ccc2c(c1)ccc1ccccc12",      -4.45),  # Phenanthrene
]

df = pd.DataFrame(data, columns=["SMILES", "logS"])
print(f"Dataset: {len(df)} molecules")
df.head()


## 2. Generate descriptors and fingerprints

We mix two feature types:
- Classical descriptors (MW, LogP, TPSA, ...)
- Morgan fingerprints (binary)


In [ ]:
def smiles_to_features(smiles, fp_size=512, radius=2):
    """Compute descriptors + Morgan fingerprint for a SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Classical descriptors
    descriptors = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.HeavyAtomCount(mol),
    ]

    # Morgan fingerprint
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fp = mfpgen.GetFingerprint(mol)
    fp_array = np.zeros(fp_size, dtype=int)
    from rdkit.DataStructs import ConvertToNumpyArray
    ConvertToNumpyArray(fp, fp_array)

    return np.concatenate([descriptors, fp_array])

# Compute features
X = np.array([smiles_to_features(s) for s in df["SMILES"]])
y = df["logS"].values

print(f"Feature matrix: {X.shape}")
print(f"Target:         {y.shape}")


## 3. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")


## 4. Train a Random Forest

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")


## 5. Visualize predictions

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, alpha=0.7, s=80, edgecolor='k')
lims = [min(y_test.min(), y_pred.min()) - 0.5,
        max(y_test.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.5, label='y = x')
ax.set_xlabel("logS observed")
ax.set_ylabel("logS predicted")
ax.set_title(f"Solubility prediction (R² = {r2:.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Feature importance

In [ ]:
# The first 8 are descriptors, then come the fingerprint bits
descriptor_names = ["MW", "LogP", "HBD", "HBA", "TPSA",
                     "RotBonds", "AromaticRings", "HeavyAtoms"]
importances = model.feature_importances_[:8]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(descriptor_names, importances, color='steelblue')
ax.set_xlabel("Importance")
ax.set_title("Importance of classical descriptors")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# As expected, LogP and MW are the top predictors of solubility:
# the more lipophilic, the less water-soluble.


## 📝 Exercises

1. **Easy**: add 5 of your own drugs and predict their logS.
2. **Medium**: try other models (`GradientBoostingRegressor`, `KNeighborsRegressor`) and compare.
3. **Medium**: download the full ESOL dataset (~1100 molecules) from MoleculeNet and re-train.
4. **Hard**: train models for other ADMET properties (BBB permeability, hERG inhibition).
5. **Pharma-focused**: predict the solubility of all drugs from one therapeutic class (e.g. statins) and rank them.

## ⚠️ Important caveats

This is an oversimplified example. In a real project you would need:

- **Much more data** (thousands, not 30 molecules)
- **Time / cluster-based splits** (not just random) to avoid leakage
- **Applicability domain** to know which predictions are trustworthy
- **Cross-validation and rigorous benchmarking**

## 📚 Resources

- [MoleculeNet](https://moleculenet.org/) — standard benchmarks for ADMET
- [DeepChem](https://deepchem.io/) — DL framework for drug discovery
- [Delaney 2004 (ESOL)](https://pubs.acs.org/doi/10.1021/ci034243x)
